In [1]:
!wget -O diabetes.zip "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
!unzip diabetes.zip

--2026-06-25 06:53:47--  https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘diabetes.zip’

diabetes.zip            [  <=>               ]   3.02M  14.9MB/s    in 0.2s    

2026-06-25 06:53:47 (14.9 MB/s) - ‘diabetes.zip’ saved [3170254]

Archive:  diabetes.zip
  inflating: diabetic_data.csv       
  inflating: IDS_mapping.csv         


In [2]:
import pandas as pd

df = pd.read_csv("diabetic_data.csv")
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1. Simple Setup
First, we create folders to keep our project organized. One for the original data and one for the clean data.

In [ ]:
import pandas as pd
import numpy as np
import os

# Create folders for organization
os.makedirs('raw_data', exist_ok=True)
os.makedirs('cleaned_data', exist_ok=True)

# Move the file to our raw folder
if os.path.exists('diabetic_data.csv'):
    os.replace('diabetic_data.csv', 'raw_data/diabetic_data.csv')

print('Folders created and data moved!')

### 2. Loading the Data
We read the file into a table (DataFrame) so we can start cleaning it.

In [ ]:
# Load the data
df = pd.read_csv('raw_data/diabetic_data.csv')

# Show the size (Rows, Columns)
print(f'Original rows: {df.shape[0]}')
df.head()

### 3. Fixing Hidden Missing Values ('?')
This data uses '?' instead of leaving cells empty. We will replace '?' with a proper empty value (NaN) so Python can find them.

In [ ]:
# Change '?' to NaN (Not a Number)
df.replace('?', np.nan, inplace=True)

# Count how many empty values we found
print('Empty values per column:')
print(df.isnull().sum().sort_values(ascending=False).head(5))

### 4. Removing Null (Empty) Values
Columns with too much missing data are not helpful, so we drop them. Then, we remove any remaining rows that still have empty cells.

In [ ]:
# Remove columns that are mostly empty
df.drop(columns=['weight', 'payer_code', 'medical_specialty'], inplace=True)

# Remove every row that has even one null (empty) value left
df.dropna(inplace=True)

print(f'Rows left after removing nulls: {df.shape[0]}')

### 5. Cleaning Age and Gender
We fix the 'gender' column and turn age ranges like '[50-60)' into a single number like 55.0.

In [ ]:
# Remove invalid gender records
df = df[df['gender'] != 'Unknown/Invalid']

# Function to calculate midpoint of age range
def fix_age(age_range):
    clean = age_range.replace('[', '').replace(')', '')
    low, high = clean.split('-')
    return (int(low) + int(high)) / 2

df['age'] = df['age'].apply(fix_age)
print('Age and Gender are now clean!')

### 6. Final Step: Saving the Clean Data
We save our final clean version for your project.

In [ ]:
# Save the cleaned data
df.to_csv('cleaned_data/diabetes_cleaned.csv', index=False)

# Create a version without ID numbers for Machine Learning
ml_data = df.drop(columns=['encounter_id', 'patient_nbr'])
ml_data.to_csv('cleaned_data/diabetes_ml_ready.csv', index=False)

print('Success! Your clean files are in the cleaned_data folder.')

### Step 1: Library Imports & Setup
We use `pandas` for tables and `numpy` for math. We also create folders to keep our project organized.

In [ ]:
import pandas as pd
import numpy as np
import os

# Create folders so we don't mix up raw and clean data
os.makedirs('raw_data', exist_ok=True)
os.makedirs('cleaned_data', exist_ok=True)

# Move the original file into the raw folder
if os.path.exists('diabetic_data.csv'):
    os.replace('diabetic_data.csv', 'raw_data/diabetic_data.csv')

print("Folders are ready!")

### Step 2: Load the Raw Data
We load our dataset into a variable called `df`. This is our working copy.

In [ ]:
# Read the CSV file
df = pd.read_csv('raw_data/diabetic_data.csv')

# Print the total rows and columns
print(f"The data has {df.shape[0]} rows and {df.shape[1]} columns.")

### Step 3: Handle Hidden Missing Values ('?')
In this data, missing info is written as '?'. We must change '?' to `NaN` (Not a Number) so that Python can count them as nulls.

In [ ]:
# Change every '?' to a proper Null value (NaN)
df.replace('?', np.nan, inplace=True)

# Count how many Null values are in each column
print("Null values found per column:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

### Step 4: Delete Null Values
We remove columns that are mostly empty (like 'weight') and then delete any row that still has a null value.

In [ ]:
# 1. Drop columns that have too many missing values
df.drop(columns=['weight', 'payer_code', 'medical_specialty'], inplace=True)

# 2. Drop all remaining rows that have ANY null values
df.dropna(inplace=True)

print(f"Rows remaining after deleting all nulls: {df.shape[0]}")

### Step 5: Fix Age and Gender
Age is currently text like '[50-60)'. We will turn it into a number (55). We also remove 'Unknown' gender rows.

In [ ]:
# Remove invalid gender entries
df = df[df['gender'] != 'Unknown/Invalid']

# A simple function to get the middle of the age range
def calculate_age(text):
    # Removes '[' and ')', then splits '50-60' into 50 and 60
    clean_text = text.replace('[', '').replace(')', '')
    low, high = clean_text.split('-')
    return (int(low) + int(high)) / 2

# Apply the function to the age column
df['age'] = df['age'].apply(calculate_age)

print("Age and Gender are now cleaned!")

### Step 6: Save the Final Clean Data
We save our work into the 'cleaned_data' folder so we can use it for our project.

In [ ]:
# Save the fully cleaned data
df.to_csv('cleaned_data/diabetes_cleaned_simple.csv', index=False)

# Save a special version for Machine Learning (removing ID columns)
ml_data = df.drop(columns=['encounter_id', 'patient_nbr'])
ml_data.to_csv('cleaned_data/diabetes_ml_ready_simple.csv', index=False)

print("Success! Your clean files are ready in the cleaned_data folder.")